# 03 SVM: Predict Penguin Species and Compare Three Models

This notebook uses an RBF-kernel SVM to predict `species` and compares Logistic Regression, Random Forest, and SVM on the same train/test split.


In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE = 0.25

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


## 1. Load Data


In [ ]:
candidate_paths = [
    Path('../../data/ml_data/penguins.csv'),
    Path('../data/ml_data/penguins.csv'),
    Path('/workspace/data/ml_data/penguins.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/penguins.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find penguins.csv. Please check data/ml_data/penguins.csv')

penguins = pd.read_csv(data_path)
print('Data path:', data_path)
print('Data shape:', penguins.shape)
penguins.head()


## 2. Prepare Training and Test Sets


In [ ]:
target_col = 'species'
drop_cols = ['Unnamed: 0']
feature_cols = [col for col in penguins.columns if col not in drop_cols + [target_col]]

X = penguins[feature_cols].copy()
y = penguins[target_col].copy()

numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'year']
categorical_features = ['island', 'sex']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Test-set class distribution:')
print(y_test.value_counts().sort_index())


## 3. Define the Three Models


In [ ]:
def make_preprocessor(scale_numeric=True):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

    return ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ])

logistic_kwargs = {'max_iter': 2000}
if 'multi_class' in inspect.signature(LogisticRegression).parameters:
    logistic_kwargs['multi_class'] = 'auto'

models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', make_preprocessor(scale_numeric=True)),
        ('model', LogisticRegression(**logistic_kwargs)),
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', make_preprocessor(scale_numeric=False)),
        ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
    'SVM (RBF)': Pipeline([
        ('preprocessor', make_preprocessor(scale_numeric=True)),
        ('model', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name} fitted')


## 4. Compare Metrics Across the Three Models


In [ ]:
comparison_rows = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average='macro',
        zero_division=0,
    )
    comparison_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
    })

comparison = pd.DataFrame(comparison_rows).sort_values('macro_f1', ascending=False)
comparison.round(4)


## 5. Detailed SVM Evaluation


In [ ]:
svm_pred = predictions['SVM (RBF)']

print('SVM Accuracy:', round(accuracy_score(y_test, svm_pred), 4))
print('\nSVM Classification report:')
print(classification_report(y_test, svm_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, svm_pred, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, cmap='Purples', colorbar=False)
ax.set_title('SVM (RBF) Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.show()

pd.DataFrame({'actual': y_test.values, 'svm_predicted': svm_pred}, index=y_test.index).head(12)


## 6. Compare the Three Confusion Matrices Side by Side


In [ ]:
labels = sorted(y.unique())
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


## 7. Summary

The three models use the same training and test sets, so accuracy, macro precision, macro recall, and macro F1 can be compared directly. For a three-class task, macro metrics give each species equal weight, which is more informative for smaller classes than accuracy alone.
